In [1]:
# =========================
# Standard library
# =========================
import sys
import math
import random
import logging
import subprocess
import warnings
import collections
from pathlib import Path

warnings.filterwarnings("ignore")

from IPython.display import display


# =========================
# Scientific computing
# =========================
import numpy as np
import pandas as pd
from scipy import stats
from scipy import __version__ as scipy_version


# =========================
# Plotting
# =========================
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter


# =========================
# Scikit-learn
# =========================
from sklearn import metrics, model_selection
from sklearn import __version__ as sklearn_version

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from sklearn.model_selection import (
    KFold,
    GroupKFold,
    GroupShuffleSplit,
    cross_validate,
    train_test_split,
)

from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import LocalOutlierFactor


# =========================
# Optimization
# =========================
import optuna


# =========================
# Gradient boosting
# =========================
try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except Exception as e:
    raise RuntimeError(
        "LightGBM not installed. Please install it with `pip install lightgbm`."
    ) from e

try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except Exception as e:
    raise RuntimeError(
        "XGBoost not installed. Please install it with `pip install xgboost`."
    ) from e


# =========================
# RNA / project utilities
# =========================
import RNA
import sylib


# =========================
# Version info
# =========================
print(f"python    = {sys.version_info[0]}.{sys.version_info[1]}.{sys.version_info[2]}")
print(f"pandas    = {pd.__version__}")
print(f"numpy     = {np.__version__}")
print(f"scipy     = {scipy_version}")
print(f"sklearn   = {sklearn_version}")
print(f"optuna    = {optuna.__version__}")
print(f"lightgbm  = {lgb.__version__}")
print(f"xgboost   = {xgb.__version__}")
print(f"ViennaRNA = {RNA.__version__}")
print(f"sylib     = {sylib.__version__}")


# =========================
# Logging
# =========================
progress_bar = sylib.utils.ProgressBar()

logging.root.handlers = []
stream_handler = logging.StreamHandler(sys.stderr)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler],
)

logger = logging.getLogger(__name__)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

python    = 3.11.15
pandas    = 2.3.3
numpy     = 2.4.6
scipy     = 1.17.1
sklearn   = 1.9.0
optuna    = 4.9.0
lightgbm  = 4.6.0
xgboost   = 3.2.0
ViennaRNA = 2.7.2
sylib     = 0.3.0.dev0+ae18bb2


In [2]:
# ============================================
# Benchmark settings
# ============================================

DATA_DIR = Path("../data/raw")

# Target and split
score_col_name = "PR_var"
group_col_name = None
test_data_ratio = 0.20

# Reproducibility and compute
random_state = 0
k_cv = 10
n_jobs = 10
n_trials = 20

# Model families
linear_models = [
    "PLS",
    "Ridge",
    "Lasso",
    "ElasticNet",
]

nonlinear_models = [
    "KernelRidge",
    "RandomForest",
    "XGBoost",
    "LightGBM",
]

model_order = linear_models + nonlinear_models

model_family_dict = {
    **{model: "Linear" for model in linear_models},
    **{model: "Nonlinear" for model in nonlinear_models},
}

# Species datasets and species-specific RNA folding temperature
species_config = {
    "AT21": {
        "file": DATA_DIR / "AT21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz",
        "rna_temp": 22,
    },
    "NB21": {
        "file": DATA_DIR / "NB21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz",
        "rna_temp": 22,
    },
    "OS21": {
        "file": DATA_DIR / "OS21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz",
        "rna_temp": 25,
    },
}

print("Species configuration:")
for species, cfg in species_config.items():
    print(f"{species}: file={cfg['file']}, rna_temp={cfg['rna_temp']}")

print("\nModel families:")
for model in model_order:
    print(f"{model}: {model_family_dict[model]}")

Species configuration:
AT21: file=../data/raw/AT21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz, rna_temp=22
NB21: file=../data/raw/NB21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz, rna_temp=22
OS21: file=../data/raw/OS21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz, rna_temp=25

Model families:
PLS: Linear
Ridge: Linear
Lasso: Linear
ElasticNet: Linear
KernelRidge: Nonlinear
RandomForest: Nonlinear
XGBoost: Nonlinear
LightGBM: Nonlinear


In [3]:
# ============================================
# Check input files
# ============================================

for species, cfg in species_config.items():
    file_path = cfg["file"]
    print(species)
    print(" ", file_path)
    print(" exists:", file_path.exists())

AT21
  ../data/raw/AT21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
 exists: True
NB21
  ../data/raw/NB21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
 exists: True
OS21
  ../data/raw/OS21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
 exists: True


In [4]:
# ============================================
# Basic sequence helpers
# ============================================

def normalize_rna(seq):
    if pd.isna(seq):
        return ""
    return str(seq).upper().replace("T", "U")


def cat_mrna(utr5, cds, utr3):
    return normalize_rna(utr5) + normalize_rna(cds) + normalize_rna(utr3)

In [5]:
# ============================================
# Model registry: default benchmark models
# ============================================

def get_default_models(random_state=0, n_jobs=10):

    linear_models = {

        "PLS": PLSRegression(
            n_components=5,
        ),

        "Ridge": Ridge(
            alpha=1.0,
        ),

        "Lasso": Lasso(
            alpha=0.001,
            max_iter=10000,
            random_state=random_state,
        ),

        "ElasticNet": ElasticNet(
            alpha=0.001,
            l1_ratio=0.5,
            max_iter=10000,
            random_state=random_state,
        ),
    }

    nonlinear_models = {

        "KernelRidge": KernelRidge(
            alpha=1.0,
            kernel="rbf",
            gamma=None,
        ),

        "RandomForest": RandomForestRegressor(
            n_estimators=300,
            random_state=random_state,
            n_jobs=n_jobs,
        ),

        "XGBoost": XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=1.0,
            colsample_bytree=1.0,
            random_state=random_state,
            n_jobs=n_jobs,
        ),

        "LightGBM": LGBMRegressor(
            objective="regression",
            n_estimators=300,
            learning_rate=0.05,
            random_state=random_state,
            n_jobs=n_jobs,
            verbosity=-1,
        ),
    }

    return {
        **linear_models,
        **nonlinear_models,
    }


default_models = get_default_models(
    random_state=random_state,
    n_jobs=n_jobs,
)

print("Linear models:")
print(list(linear_models))

print("\nNonlinear models:")
print(list(nonlinear_models))

print("\nBenchmark order:")
print(list(default_models))

Linear models:
['PLS', 'Ridge', 'Lasso', 'ElasticNet']

Nonlinear models:
['KernelRidge', 'RandomForest', 'XGBoost', 'LightGBM']

Benchmark order:
['PLS', 'Ridge', 'Lasso', 'ElasticNet', 'KernelRidge', 'RandomForest', 'XGBoost', 'LightGBM']


In [6]:
# ============================================
# Load one species
# ============================================

def load_species_data(species, species_config):

    file_path = species_config[species]["file"]

    seq_df, metadata = sylib.fileio.load_df(
        str(file_path)
    )

    metadata.print_minimum_data(
        label=species,
        logger=logger,
        logging_level="info",
    )

    return seq_df, metadata

In [7]:
# ============================================
# Feature Evaluation: Definition of Features
# Only benchmark features are included.
# ============================================

def eval_length(seq_list, region_name, logger=logger):
    return pd.DataFrame({
        f"{region_name}.Length": [
            len(seq) if isinstance(seq, str) else 0
            for seq in seq_list
        ]
    })


def eval_kmer_freq(seq_list, region_name, is_rna_seq=True, k=1, logger=logger):
    count_dict = sylib.sequtils.count_nuc_k_mer_of_seq_list(
        seq_list,
        is_rna_seq=is_rna_seq,
        k=k,
    )

    feat_df = pd.DataFrame({
        f"{region_name}.{key}-freq": count_dict[key]
        for key in count_dict
    })

    seq_len_sr = pd.Series([
        len(seq) if isinstance(seq, str) else 0
        for seq in seq_list
    ])

    seq_len_sr[seq_len_sr == 0] = 1

    for col_name in feat_df.columns:
        feat_df[col_name] /= seq_len_sr
        feat_df[col_name] *= 1000

    return feat_df


def eval_s_freq(seq_list, region_name, logger=logger):
    v = []
    for seq in seq_list:
        seq = seq if isinstance(seq, str) else ""
        if len(seq) == 0:
            v.append(0.0)
        else:
            v.append((seq.count("G") + seq.count("C")) / len(seq) * 1000)

    return pd.DataFrame({f"{region_name}.S-freq": v})


def eval_y_freq(seq_list, region_name, logger=logger):
    v = []
    for seq in seq_list:
        seq = seq if isinstance(seq, str) else ""
        if len(seq) == 0:
            v.append(0.0)
        else:
            v.append((seq.count("T") + seq.count("U") + seq.count("C")) / len(seq) * 1000)

    return pd.DataFrame({f"{region_name}.Y-freq": v})


def eval_k_freq(seq_list, region_name, logger=logger):
    v = []
    for seq in seq_list:
        seq = seq if isinstance(seq, str) else ""
        if len(seq) == 0:
            v.append(0.0)
        else:
            v.append((seq.count("G") + seq.count("T") + seq.count("U")) / len(seq) * 1000)

    return pd.DataFrame({f"{region_name}.K-freq": v})


def eval_n_uaug(utr5_seq_list, region_name, is_rna_seq=True, logger=logger):
    if is_rna_seq:
        return pd.DataFrame({
            f"{region_name}.uAUG": [
                seq.count("AUG") if isinstance(seq, str) else 0
                for seq in utr5_seq_list
            ]
        })
    else:
        return pd.DataFrame({
            f"{region_name}.uATG": [
                seq.count("ATG") if isinstance(seq, str) else 0
                for seq in utr5_seq_list
            ]
        })


def eval_n_uorf(utr5_seq_list, region_name, is_rna_seq=True, logger=logger):
    feat_df_dict = {
        f"{region_name}.uORF-NO": [],
        f"{region_name}.uORF-IO": [],
        f"{region_name}.uORF-OO": [],
    }

    if is_rna_seq:
        start_codon = "AUG"
        stop_codons = {"UAA", "UGA", "UAG"}
    else:
        start_codon = "ATG"
        stop_codons = {"TAA", "TGA", "TAG"}

    for seq in utr5_seq_list:
        seq = seq if isinstance(seq, str) else ""
        L = len(seq)

        n_no = 0
        n_io = 0
        n_oo = 0

        for i in range(L - 2):
            if seq[i:i+3] == start_codon:
                has_stop = False

                for j in range(i + 3, L - 2, 3):
                    if seq[j:j+3] in stop_codons:
                        has_stop = True
                        break

                if has_stop:
                    n_no += 1
                elif (L - i) % 3 == 0:
                    n_io += 1
                else:
                    n_oo += 1

        feat_df_dict[f"{region_name}.uORF-NO"].append(n_no)
        feat_df_dict[f"{region_name}.uORF-IO"].append(n_io)
        feat_df_dict[f"{region_name}.uORF-OO"].append(n_oo)

    return pd.DataFrame(feat_df_dict)


def eval_n_daug(utr3_seq_list, region_name, is_rna_seq=True, logger=logger):
    if is_rna_seq:
        return pd.DataFrame({
            f"{region_name}.dAUG": [
                seq.count("AUG") if isinstance(seq, str) else 0
                for seq in utr3_seq_list
            ]
        })
    else:
        return pd.DataFrame({
            f"{region_name}.dATG": [
                seq.count("ATG") if isinstance(seq, str) else 0
                for seq in utr3_seq_list
            ]
        })


def eval_n_dorf(utr3_seq_list, region_name, is_rna_seq=True, logger=logger):
    feat_df_dict = {
        f"{region_name}.dORF": [],
        f"{region_name}.dORF-NS": [],
    }

    if is_rna_seq:
        start_codon = "AUG"
        stop_codons = {"UAA", "UGA", "UAG"}
    else:
        start_codon = "ATG"
        stop_codons = {"TAA", "TGA", "TAG"}

    for seq in utr3_seq_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq + "AA"
        L = len(seq)

        n_no = 0
        n_ns = 0

        for i in range(L - 2):
            if seq[i:i+3] == start_codon:
                has_stop = False

                for j in range(i + 3, L - 2, 3):
                    if seq[j:j+3] in stop_codons:
                        has_stop = True
                        break

                if has_stop:
                    n_no += 1
                else:
                    n_ns += 1

        feat_df_dict[f"{region_name}.dORF"].append(n_no)
        feat_df_dict[f"{region_name}.dORF-NS"].append(n_ns)

    return pd.DataFrame(feat_df_dict)

# ============================================
# Feature Evaluation: CDS coding features
# Only benchmark non-MFE features are included here.
# MFE is precomputed separately by script.
# ============================================

def eval_codon_usage(cds_seq_list, region_name, is_rna_seq=True, should_remove_stop_codons=False, logger=logger):
    mer3_list = sylib.sequtils.get_k_mer_nuc_list(3, is_rna_seq=is_rna_seq)
    aa_codon_dict = sylib.sequtils.get_aa_codon_dict(
        is_rna_seq=is_rna_seq,
        code_type="3-letter",
    )

    n_codons_dict = {mer3: [0] * len(cds_seq_list) for mer3 in mer3_list}

    for i, cds_seq in enumerate(cds_seq_list):
        cds_seq = cds_seq if isinstance(cds_seq, str) else ""
        for j in range(0, len(cds_seq), 3):
            codon = cds_seq[j:j+3]
            if codon in n_codons_dict:
                n_codons_dict[codon][i] += 1

    codons_df = pd.DataFrame(n_codons_dict)
    usage_df_dict = {}

    for aa, codon_list in aa_codon_dict.items():
        aa_sr = pd.Series([0] * len(cds_seq_list))

        for codon in codon_list:
            aa_sr += codons_df[codon]

        aa_sr[aa_sr == 0] = 1

        for codon in codon_list:
            key = f"{region_name}.{aa}-{codon}"
            usage_df_dict[key] = (codons_df[codon] / aa_sr) * 1000

    if is_rna_seq:
        usage_df_dict.pop(f"{region_name}.Met-AUG", None)
        usage_df_dict.pop(f"{region_name}.Trp-UGG", None)

        if should_remove_stop_codons:
            for c in ("UAA", "UGA", "UAG"):
                usage_df_dict.pop(f"{region_name}.Ter-{c}", None)
    else:
        usage_df_dict.pop(f"{region_name}.Met-ATG", None)
        usage_df_dict.pop(f"{region_name}.Trp-TGG", None)

        if should_remove_stop_codons:
            for c in ("TAA", "TGA", "TAG"):
                usage_df_dict.pop(f"{region_name}.Ter-{c}", None)

    return pd.DataFrame(usage_df_dict)


def wobble_freq(cds_list, region_label="CDS"):
    alphabet = ["A", "U", "G", "C"]
    cols = {f"{region_label}.wobble_pct_{b}": [] for b in alphabet}

    for seq in cds_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq.upper().replace("T", "U")

        wobble_bases = [
            seq[i]
            for i in range(2, len(seq), 3)
            if i < len(seq)
        ]

        total = len(wobble_bases)

        for b in alphabet:
            cols[f"{region_label}.wobble_pct_{b}"].append(
                wobble_bases.count(b) / total if total > 0 else 0.0
            )

    return pd.DataFrame(cols)


CODON_TO_AA_RNA = {
    "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L",
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S",
    "UAU": "Y", "UAC": "Y", "UAA": "X", "UAG": "X",
    "UGU": "C", "UGC": "C", "UGA": "X", "UGG": "W",
    "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AUU": "I", "AUC": "I", "AUA": "I", "AUG": "M",
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGU": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}

AA_LIST = sorted(set(CODON_TO_AA_RNA.values()))


def aa_freq_from_cds(cds_list, region_label="CDS"):
    cols = {f"{region_label}.aa_pct_{aa}": [] for aa in AA_LIST}

    for seq in cds_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq.upper().replace("T", "U")

        counts = {aa: 0 for aa in AA_LIST}
        total = 0

        for i in range(0, len(seq) - 2, 3):
            codon = seq[i:i+3]
            aa = CODON_TO_AA_RNA.get(codon)

            if aa is None:
                continue

            counts[aa] += 1
            total += 1

        for aa in AA_LIST:
            cols[f"{region_label}.aa_pct_{aa}"].append(
                counts[aa] / total if total > 0 else 0.0
            )

    return pd.DataFrame(cols)


DICODONS_DNA = [
    "AGGCGA", "AGGCGG", "ATACGA", "ATACGG", "CGAATA",
    "CGACCG", "CGACGA", "CGACGG", "CGACTG", "CGAGCG",
    "CTCATA", "CTCCCG", "CTGATA", "CTGCCG", "CTGCGA",
    "CTGCTG", "CTTCTG", "GTACCG", "GTACGA", "GTGCGA",
]

DICODONS_RNA = [d.replace("T", "U") for d in DICODONS_DNA]


def dicodon_counts_rna(cds_list, region_label="CDS", normalized=False):
    cols = {f"{region_label}.dicodon_{d}": [] for d in DICODONS_RNA}

    for seq in cds_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq.upper().replace("T", "U")

        counts = {d: 0 for d in DICODONS_RNA}
        total = 0

        for i in range(0, len(seq) - 5, 3):
            dicodon = seq[i:i+6]

            if len(dicodon) == 6:
                total += 1

                if dicodon in counts:
                    counts[dicodon] += 1

        for d in DICODONS_RNA:
            value = counts[d] / total if (normalized and total > 0) else counts[d]
            cols[f"{region_label}.dicodon_{d}"].append(value)

    return pd.DataFrame(cols)

In [8]:
# ============================================
# Feature Evaluation: Execution of non-MFE features
# ============================================

def build_non_mfe_features(df):
    parts = []

    # 1. Region length
    parts += [
        eval_length(df["5'UTR"], "5'UTR"),
        eval_length(df["CDS"],   "CDS"),
        eval_length(df["3'UTR"], "3'UTR"),
    ]

    # 2. Region nucleotide / k-mer composition
    for k in [1, 2, 3]:
        parts += [
            eval_kmer_freq(df["5'UTR"], "5'UTR", is_rna_seq=True, k=k),
            eval_kmer_freq(df["CDS"],   "CDS",   is_rna_seq=True, k=k),
            eval_kmer_freq(df["3'UTR"], "3'UTR", is_rna_seq=True, k=k),
        ]

    # 3. Degenerate nucleotide class features
    parts += [
        eval_s_freq(df["5'UTR"], "5'UTR"),
        eval_y_freq(df["5'UTR"], "5'UTR"),
        eval_k_freq(df["5'UTR"], "5'UTR"),
        eval_s_freq(df["3'UTR"], "3'UTR"),
        eval_y_freq(df["3'UTR"], "3'UTR"),
        eval_k_freq(df["3'UTR"], "3'UTR"),
    ]

    # 4. Upstream/downstream ORF-related features
    parts += [
        eval_n_uaug(df["5'UTR"], "5'UTR", is_rna_seq=True),
        eval_n_uorf(df["5'UTR"], "5'UTR", is_rna_seq=True),
        eval_n_daug(df["3'UTR"], "3'UTR", is_rna_seq=True),
        eval_n_dorf(df["3'UTR"], "3'UTR", is_rna_seq=True),
    ]

    # 5. CDS coding features
    parts += [
        eval_codon_usage(df["CDS"], "CDS", is_rna_seq=True, should_remove_stop_codons=True),
        wobble_freq(df["CDS"], "CDS"),
        aa_freq_from_cds(df["CDS"], "CDS"),
        dicodon_counts_rna(df["CDS"], "CDS", normalized=True),
    ]

    x_feat_df = pd.concat(parts, axis=1)

    return x_feat_df

In [9]:
# ============================================
# MFE precompute wrapper
# ============================================

def get_mfe_file_path(seq_file):
    seq_file = Path(seq_file)
    return seq_file.with_name(
        seq_file.name.replace(".RNA.seq_data.tsv.gz", ".MFE.tsv.gz")
    )


def ensure_mfe_file(seq_file, temperature, n_workers=10, force=False):
    seq_file = Path(seq_file)
    mfe_file = get_mfe_file_path(seq_file)

    if mfe_file.exists() and not force:
        logger.info("MFE file already exists: %s", mfe_file)
        return mfe_file

    cmd = [
        sys.executable,
        "../scripts/precompute_mfe.py",
        str(seq_file),
        str(mfe_file),
        "--temperature",
        str(temperature),
        "-p",
        str(n_workers),
    ]

    logger.info("Running MFE script:")
    logger.info(" ".join(cmd))

    subprocess.run(cmd, check=True)

    return mfe_file

In [10]:
# ============================================
# Build complete feature matrix for one species
# ============================================

def build_species_feature_matrix(species, species_config, n_workers=10, force_mfe=False):
    logger.info("Building feature matrix for %s", species)

    seq_df, metadata = load_species_data(species, species_config)

    # non-MFE features
    x_non_mfe_df = build_non_mfe_features(seq_df)

    # MFE features
    mfe_file = ensure_mfe_file(
        seq_file=species_config[species]["file"],
        temperature=species_config[species]["rna_temp"],
        n_workers=n_workers,
        force=force_mfe,
    )

    mfe_df, mfe_metadata = sylib.fileio.load_df(str(mfe_file))

    # safety check
    assert (seq_df["var_id"].values == mfe_df["var_id"].values).all()

    mfe_feature_cols = [
        "MFE.5'UTR",
        "MFE.CDS",
        "MFE.3'UTR",
        "MFE.mRNA",
    ]

    x_feat_df = pd.concat(
        [
            x_non_mfe_df.reset_index(drop=True),
            mfe_df[mfe_feature_cols].reset_index(drop=True),
        ],
        axis=1,
    )

    # quality checks
    n_missing = x_feat_df.isna().sum().sum()
    n_infinite = np.isinf(x_feat_df.to_numpy()).sum()
    n_duplicated = x_feat_df.columns.duplicated().sum()

    logger.info(
        "%s features: shape=%s, missing=%s, infinite=%s, duplicated_cols=%s",
        species,
        x_feat_df.shape,
        n_missing,
        n_infinite,
        n_duplicated,
    )

    if n_missing > 0:
        raise ValueError(f"{species}: missing values found in feature matrix")
    if n_infinite > 0:
        raise ValueError(f"{species}: infinite values found in feature matrix")
    if n_duplicated > 0:
        raise ValueError(f"{species}: duplicated feature columns found")

    y_array = np.array(seq_df[score_col_name])

    return seq_df, x_feat_df, y_array

In [11]:
# ============================================
# Test feature matrix generation for all species
# ============================================

species_feature_data = {}

for species in species_config:
    seq_df_sp, x_feat_df_sp, y_array_sp = build_species_feature_matrix(
        species,
        species_config,
        n_workers=n_jobs,
        force_mfe=False,
    )

    species_feature_data[species] = {
        "seq_df": seq_df_sp,
        "x_feat_df": x_feat_df_sp,
        "y_array": y_array_sp,
    }

    print(species, seq_df_sp.shape, x_feat_df_sp.shape, y_array_sp.shape)

2026-06-18 16:26:39,109     INFO: Building feature matrix for AT21
2026-06-18 16:26:39,234     INFO: AT21 |   name = AT21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
2026-06-18 16:26:39,235     INFO: AT21 |    md5 = d66042fbcc8c89972a7d978665f0d63f
2026-06-18 16:26:39,236     INFO: AT21 | md5_gz = 32f956b63477c6726ca680a8317ad7e9
2026-06-18 16:26:40,516     INFO: MFE file already exists: ../data/raw/AT21.PR_var.N50-TSL0-ARL0.MFE.tsv.gz
2026-06-18 16:26:40,552     INFO: AT21 features: shape=(7171, 376), missing=0, infinite=0, duplicated_cols=0
2026-06-18 16:26:40,553     INFO: Building feature matrix for NB21
2026-06-18 16:26:40,642     INFO: NB21 |   name = NB21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
2026-06-18 16:26:40,643     INFO: NB21 |    md5 = c5f202c7acd9e921077734ffdaa258a8
2026-06-18 16:26:40,643     INFO: NB21 | md5_gz = ec4b2215377cafd2b9766c32c0db5f41


AT21 (7171, 11) (7171, 376) (7171,)


2026-06-18 16:26:41,675     INFO: MFE file already exists: ../data/raw/NB21.PR_var.N50-TSL0-ARL0.MFE.tsv.gz
2026-06-18 16:26:41,695     INFO: NB21 features: shape=(5429, 376), missing=0, infinite=0, duplicated_cols=0
2026-06-18 16:26:41,696     INFO: Building feature matrix for OS21
2026-06-18 16:26:41,788     INFO: OS21 |   name = OS21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
2026-06-18 16:26:41,788     INFO: OS21 |    md5 = 1d465f412dab793a8b9d606b44ed4d7e
2026-06-18 16:26:41,789     INFO: OS21 | md5_gz = 9f5fe7673afaf4c125128c3212505fda


NB21 (5429, 11) (5429, 376) (5429,)


2026-06-18 16:26:42,937     INFO: MFE file already exists: ../data/raw/OS21.PR_var.N50-TSL0-ARL0.MFE.tsv.gz
2026-06-18 16:26:42,959     INFO: OS21 features: shape=(5599, 376), missing=0, infinite=0, duplicated_cols=0


OS21 (5599, 11) (5599, 376) (5599,)


In [12]:
# ============================================
# Feature matrix summary across species
# ============================================

summary_rows = []

for species, data in species_feature_data.items():
    x_feat_df = data["x_feat_df"]

    summary_rows.append({
        "species": species,
        "n_samples": x_feat_df.shape[0],
        "n_features": x_feat_df.shape[1],
        "n_missing": x_feat_df.isna().sum().sum(),
        "n_infinite": np.isinf(x_feat_df.to_numpy()).sum(),
        "n_duplicated_cols": x_feat_df.columns.duplicated().sum(),
        "n_constant_features": (x_feat_df.nunique() <= 1).sum(),
    })

feature_summary_df = pd.DataFrame(summary_rows)

display(feature_summary_df)

,species,n_samples,n_features,n_missing,n_infinite,n_duplicated_cols,n_constant_features
0,AT21,7171,376,0,0,0,0
1,NB21,5429,376,0,0,0,0
2,OS21,5599,376,0,0,0,0


In [13]:
# ============================================
# Preprocessing: outlier removal, transform
# ============================================

xform_method = "yeo-johnson"
outlier_removal_method = "Z-score"
z_score_expected_n = 0.5
lof_contamination = "auto"

outlier_det_col_list = [
    "5'UTR.Length",
    "CDS.Length",
    "3'UTR.Length",
]

def detect_outliers(
    xformed_train_array,
    xformed_test_array,
    removal_method,
    z_score_expected_n=0.5,
    lof_contamination="auto",
):
    if removal_method is None:
        train_bool_array = np.ones(len(xformed_train_array), dtype=bool)
        test_bool_array = np.ones(len(xformed_test_array), dtype=bool)

    elif removal_method == "Z-score":
        min_thresh = stats.norm.ppf(
            (z_score_expected_n / 2) / len(xformed_train_array),
            loc=0,
            scale=1,
        )
        max_thresh = -min_thresh

        train_bool_array = (
            (xformed_train_array >= min_thresh)
            & (xformed_train_array <= max_thresh)
        )
        test_bool_array = (
            (xformed_test_array >= min_thresh)
            & (xformed_test_array <= max_thresh)
        )

    elif removal_method == "LOF":
        clf = LocalOutlierFactor(
            novelty=True,
            contamination=lof_contamination,
        ).fit(xformed_train_array.reshape(-1, 1))

        train_bool_array = clf.predict(xformed_train_array.reshape(-1, 1)) == 1
        test_bool_array = clf.predict(xformed_test_array.reshape(-1, 1)) == 1

    else:
        raise ValueError(f"Unknown outlier removal method: {removal_method}")

    return train_bool_array, test_bool_array

In [14]:
# ============================================
# Prepare train/test data for benchmarking
# ============================================

def prepare_train_test_data(
    seq_df,
    x_feat_df,
    y_array,
    group_col_name=None,
    test_data_ratio=0.2,
    random_state=0,
):
    # split
    if group_col_name is None:
        n_data = len(seq_df)
        n_test = int(round(n_data * test_data_ratio))

        train_idx_array, test_idx_array = model_selection.train_test_split(
            np.arange(n_data),
            random_state=random_state,
            test_size=n_test,
        )

        g_train, g_test = None, None

    else:
        g_sr = seq_df[group_col_name]
        gss = model_selection.GroupShuffleSplit(
            n_splits=1,
            test_size=test_data_ratio,
            random_state=random_state,
        )

        train_idx_array, test_idx_array = list(
            gss.split(np.arange(len(g_sr)), groups=g_sr)
        )[0]

        g_train = g_sr.iloc[train_idx_array].copy()
        g_test = g_sr.iloc[test_idx_array].copy()

    y_train_raw = y_array[train_idx_array].copy()
    y_test_raw = y_array[test_idx_array].copy()

    x_train_raw = x_feat_df.iloc[train_idx_array, :].copy()
    x_test_raw = x_feat_df.iloc[test_idx_array, :].copy()

    # y transform for outlier detection
    pre_y_train, y_xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
        y_train_raw,
        xform_method=xform_method,
        should_stdzn=True,
    )
    pre_y_test = sylib.utils.xform_array(y_test_raw, **y_xform_kwargs)

    train_bool_array, test_bool_array = detect_outliers(
        pre_y_train,
        pre_y_test,
        outlier_removal_method,
        z_score_expected_n=z_score_expected_n,
        lof_contamination=lof_contamination,
    )

    # outlier detection using selected X columns
    for col_name in outlier_det_col_list:
        if col_name not in x_train_raw.columns:
            continue

        if x_train_raw[col_name].var() == 0:
            continue

        xformed_train_array, xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
            x_train_raw[col_name],
            xform_method=xform_method,
            should_stdzn=True,
        )
        xformed_test_array = sylib.utils.xform_array(
            x_test_raw[col_name],
            **xform_kwargs,
        )

        temp_train_bool_array, temp_test_bool_array = detect_outliers(
            xformed_train_array,
            xformed_test_array,
            outlier_removal_method,
            z_score_expected_n=z_score_expected_n,
            lof_contamination=lof_contamination,
        )

        train_bool_array &= temp_train_bool_array
        test_bool_array &= temp_test_bool_array

    # remove outliers
    y_train_raw = y_train_raw[train_bool_array]
    y_test_raw = y_test_raw[test_bool_array]

    x_train = x_train_raw[train_bool_array].copy()
    x_test = x_test_raw[test_bool_array].copy()

    if group_col_name is not None:
        g_train = g_train.iloc[train_bool_array].copy()
        g_test = g_test.iloc[test_bool_array].copy()

    # final y transform after outlier removal
    y_train, y_xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
        y_train_raw,
        xform_method=xform_method,
        should_stdzn=True,
    )
    y_test = sylib.utils.xform_array(y_test_raw, **y_xform_kwargs)

    # transform X using training-derived parameters
    removed_features = []
    x_xform_param_dict = {}

    for col_name in list(x_train.columns):
        if x_train[col_name].var() == 0:
            del x_train[col_name]
            del x_test[col_name]
            x_xform_param_dict[col_name] = None
            removed_features.append(col_name)
            continue

        if len(set(x_train[col_name])) == 2:
            x_train[col_name], xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
                x_train[col_name],
                xform_method=None,
                should_stdzn=True,
            )
        else:
            x_train[col_name], xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
                x_train[col_name],
                xform_method=xform_method,
                should_stdzn=True,
            )

        x_test[col_name] = sylib.utils.xform_array(x_test[col_name], **xform_kwargs)
        x_xform_param_dict[col_name] = xform_kwargs

    info = {
        "n_before": len(seq_df),
        "n_train_before_outlier": len(x_train_raw),
        "n_test_before_outlier": len(x_test_raw),
        "n_train_after_outlier": len(x_train),
        "n_test_after_outlier": len(x_test),
        "n_features_before_filter": x_feat_df.shape[1],
        "n_features_after_filter": x_train.shape[1],
        "n_removed_train_outliers": len(x_train_raw) - len(x_train),
        "n_removed_test_outliers": len(x_test_raw) - len(x_test),
        "n_removed_features": len(removed_features),
        "removed_features": removed_features,
    }

    return {
        "x_train": x_train,
        "x_test": x_test,
        "y_train": y_train,
        "y_test": y_test,
        "g_train": g_train,
        "g_test": g_test,
        "info": info,
    }

In [17]:
# ============================================
# Prepare train/test data for all species
# ============================================

prepared_data = {}
prep_summary_rows = []

for species, data in species_feature_data.items():
    prepared = prepare_train_test_data(
        seq_df=data["seq_df"],
        x_feat_df=data["x_feat_df"],
        y_array=data["y_array"],
        group_col_name=group_col_name,
        test_data_ratio=test_data_ratio,
        random_state=random_state,
    )

    prepared_data[species] = prepared

    row = {"species": species}
    row.update(prepared["info"])
    prep_summary_rows.append(row)

prep_summary_df = pd.DataFrame(prep_summary_rows)

display(prep_summary_df)

,species,n_before,n_train_before_outlier,n_test_before_outlier,n_train_after_outlier,n_test_after_outlier,n_features_before_filter,n_features_after_filter,n_removed_train_outliers,n_removed_test_outliers,n_removed_features,removed_features
0,AT21,7171,5737,1434,5698,1428,376,376,39,6,0,[]
1,NB21,5429,4343,1086,4305,1076,376,376,38,10,0,[]
2,OS21,5599,4479,1120,4445,1107,376,375,34,13,1,[CDS.dicodon_GUACGA]


In [15]:
# ============================================
# Model evaluation helper
# ============================================

def evaluate_predictions(y_true, y_pred):
    pearson_r, _ = stats.pearsonr(y_true, y_pred)

    return {
        "R2": metrics.r2_score(y_true, y_pred),
        "Pearson_r": pearson_r,
        "RMSE": np.sqrt(metrics.mean_squared_error(y_true, y_pred)),
        "MAE": metrics.mean_absolute_error(y_true, y_pred),
    }


# ============================================
# Run one default model
# ============================================

def run_default_model(species, model_name, model, prepared, k_cv=10, n_jobs=10):
    x_train = prepared["x_train"]
    x_test = prepared["x_test"]
    y_train = prepared["y_train"]
    y_test = prepared["y_test"]
    g_train = prepared["g_train"]

    if g_train is None:
        effective_k_cv = min(k_cv, len(y_train))

        cv = model_selection.KFold(
            n_splits=effective_k_cv,
            shuffle=True,
            random_state=random_state,
        )
        cv_groups = None

    else:
        effective_k_cv = min(k_cv, len(set(g_train)))

        cv = model_selection.GroupKFold(
            n_splits=effective_k_cv,
        )
        cv_groups = g_train

    cv_scores = model_selection.cross_validate(
        model,
        x_train,
        y_train,
        cv=cv,
        groups=cv_groups,
        scoring="r2",
        n_jobs=n_jobs,
        return_train_score=False,
    )

    model.fit(x_train, y_train)

    pred_train = model.predict(x_train)
    pred_test = model.predict(x_test)

    train_metrics = evaluate_predictions(y_train, pred_train)
    test_metrics = evaluate_predictions(y_test, pred_test)

    return {
        "species": species,
        "model": model_name,
        "model_family": model_family_dict[model_name],
        "setting": "default",
        "CV_R2": cv_scores["test_score"].mean(),
        "CV_R2_sd": cv_scores["test_score"].std(),
        "Train_R2": train_metrics["R2"],
        "Test_R2": test_metrics["R2"],
        "Train_Pearson_r": train_metrics["Pearson_r"],
        "Test_Pearson_r": test_metrics["Pearson_r"],
        "Train_RMSE": train_metrics["RMSE"],
        "Test_RMSE": test_metrics["RMSE"],
        "Train_MAE": train_metrics["MAE"],
        "Test_MAE": test_metrics["MAE"],
    }

In [18]:
# ============================================
# Run default benchmark: all species × all models
# ============================================

default_results = []

for species in species_config:
    for model_name, model in default_models.items():

        logger.info(
            "Running default model: species=%s, model=%s",
            species,
            model_name,
        )

        result = run_default_model(
            species=species,
            model_name=model_name,
            model=model,
            prepared=prepared_data[species],
            k_cv=k_cv,
            n_jobs=n_jobs,
        )

        default_results.append(result)

default_results_df = pd.DataFrame(default_results)

display(default_results_df)

display(
    default_results_df.sort_values(
        ["species", "Test_R2"],
        ascending=[True, False],
    )
)

2026-06-18 16:35:44,161     INFO: Running default model: species=AT21, model=PLS
2026-06-18 16:35:47,094     INFO: Running default model: species=AT21, model=Ridge
2026-06-18 16:35:47,542     INFO: Running default model: species=AT21, model=Lasso
2026-06-18 16:35:51,984     INFO: Running default model: species=AT21, model=ElasticNet
2026-06-18 16:36:05,335     INFO: Running default model: species=AT21, model=KernelRidge
2026-06-18 16:36:09,260     INFO: Running default model: species=AT21, model=RandomForest
2026-06-18 16:37:25,490     INFO: Running default model: species=AT21, model=XGBoost
2026-06-18 16:37:46,616     INFO: Running default model: species=AT21, model=LightGBM
2026-06-18 16:38:08,550     INFO: Running default model: species=NB21, model=PLS
2026-06-18 16:38:08,878     INFO: Running default model: species=NB21, model=Ridge
2026-06-18 16:38:09,191     INFO: Running default model: species=NB21, model=Lasso
2026-06-18 16:38:12,163     INFO: Running default model: species=NB2

,species,model,model_family,setting,CV_R2,CV_R2_sd,Train_R2,Test_R2,Train_Pearson_r,Test_Pearson_r,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE
0,AT21,PLS,Linear,default,0.152271,0.034206,0.217031,0.178716,0.465866,0.424369,0.884855,0.911727,0.686262,0.716301
1,AT21,Ridge,Linear,default,0.170467,0.034620,0.288948,0.198109,0.537545,0.452137,0.843239,0.900898,0.652107,0.701203
2,AT21,Lasso,Linear,default,0.180112,0.031340,0.274173,0.199704,0.524158,0.449291,0.851955,0.900002,0.657953,0.703148
3,AT21,ElasticNet,Linear,default,0.177490,0.032535,0.281790,0.202450,0.531160,0.453159,0.847473,0.898457,0.654547,0.700453
4,AT21,KernelRidge,Nonlinear,default,0.243076,0.037023,0.511295,0.282991,0.733514,0.533201,0.699074,0.851884,0.537384,0.652194
5,AT21,RandomForest,Nonlinear,default,0.218781,0.038602,0.895073,0.273023,0.962327,0.541205,0.323924,0.857785,0.244845,0.642146
6,AT21,XGBoost,Nonlinear,default,0.223709,0.040126,0.762430,0.277110,0.892955,0.535545,0.487412,0.855371,0.378640,0.644559
7,AT21,LightGBM,Nonlinear,default,0.225811,0.043816,0.736431,0.278224,0.874144,0.537620,0.513390,0.854711,0.393311,0.644343
8,NB21,PLS,Linear,default,0.219290,0.026906,0.295425,0.184457,0.543530,0.441240,0.839390,0.919915,0.649416,0.710536
9,NB21,Ridge,Linear,default,0.232027,0.028416,0.378337,0.235310,0.615106,0.498272,0.788456,0.890773,0.609575,0.683507


,species,model,model_family,setting,CV_R2,CV_R2_sd,Train_R2,Test_R2,Train_Pearson_r,Test_Pearson_r,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE
4,AT21,KernelRidge,Nonlinear,default,0.243076,0.037023,0.511295,0.282991,0.733514,0.533201,0.699074,0.851884,0.537384,0.652194
7,AT21,LightGBM,Nonlinear,default,0.225811,0.043816,0.736431,0.278224,0.874144,0.537620,0.513390,0.854711,0.393311,0.644343
6,AT21,XGBoost,Nonlinear,default,0.223709,0.040126,0.762430,0.277110,0.892955,0.535545,0.487412,0.855371,0.378640,0.644559
5,AT21,RandomForest,Nonlinear,default,0.218781,0.038602,0.895073,0.273023,0.962327,0.541205,0.323924,0.857785,0.244845,0.642146
3,AT21,ElasticNet,Linear,default,0.177490,0.032535,0.281790,0.202450,0.531160,0.453159,0.847473,0.898457,0.654547,0.700453
2,AT21,Lasso,Linear,default,0.180112,0.031340,0.274173,0.199704,0.524158,0.449291,0.851955,0.900002,0.657953,0.703148
1,AT21,Ridge,Linear,default,0.170467,0.034620,0.288948,0.198109,0.537545,0.452137,0.843239,0.900898,0.652107,0.701203
0,AT21,PLS,Linear,default,0.152271,0.034206,0.217031,0.178716,0.465866,0.424369,0.884855,0.911727,0.686262,0.716301
12,NB21,KernelRidge,Nonlinear,default,0.296617,0.027706,0.543173,0.282354,0.751413,0.534208,0.675890,0.862938,0.515381,0.652284
13,NB21,RandomForest,Nonlinear,default,0.271354,0.034903,0.901810,0.273894,0.963151,0.541141,0.313352,0.868009,0.234637,0.653954


In [19]:
# ============================================
# Tuning helpers
# ============================================

def get_cv(prepared, k_cv=10, random_state=0):
    g_train = prepared["g_train"]
    n_train = len(prepared["y_train"])

    effective_k_cv = min(k_cv, n_train)

    if effective_k_cv < 2:
        raise ValueError(
            f"Not enough training samples for CV: n_train={n_train}"
        )

    if g_train is None:
        cv = model_selection.KFold(
            n_splits=effective_k_cv,
            shuffle=True,
            random_state=random_state,
        )
        cv_groups = None

    else:
        random.seed(random_state)
        np.random.seed(random_state)

        n_groups = len(set(g_train))
        effective_k_cv = min(effective_k_cv, n_groups)

        if effective_k_cv < 2:
            raise ValueError(
                f"Not enough groups for GroupKFold: n_groups={n_groups}"
            )

        cv = model_selection.GroupKFold(
            n_splits=effective_k_cv,
        )
        cv_groups = g_train

    return cv, cv_groups


def collect_optuna_trials(study):

    rows = []

    for trial in study.get_trials():

        row = {
            "trial": trial.number,
            "value": trial.value,
            "state": str(trial.state),
        }

        row.update(trial.params)
        rows.append(row)

    trial_df = pd.DataFrame(rows)

    if len(trial_df) > 0 and "value" in trial_df.columns:
        trial_df = trial_df.sort_values(
            "value",
            ascending=False,
        )

    return trial_df


# ============================================
# Tunable model registry
# ============================================

def get_tuned_model(
    model_name,
    trial,
    prepared=None,
    random_state=0,
    n_jobs=10,
):

    if prepared is not None:
        n_train = prepared["x_train"].shape[0]
        n_features = prepared["x_train"].shape[1]
    else:
        n_train = 100
        n_features = 100


    # =====================================
    # Linear models
    # =====================================

    if model_name == "PLS":

        max_components = max(
            2,
            min(30, n_train - 1, n_features),
        )

        model = PLSRegression(
            n_components=trial.suggest_int(
                "n_components",
                2,
                max_components,
            ),
        )


    elif model_name == "Ridge":

        model = Ridge(
            alpha=trial.suggest_float(
                "alpha",
                1e-4,
                1e4,
                log=True,
            ),
        )


    elif model_name == "Lasso":

        model = Lasso(
            alpha=trial.suggest_float(
                "alpha",
                1e-5,
                1e1,
                log=True,
            ),
            max_iter=20000,
            random_state=random_state,
        )


    elif model_name == "ElasticNet":

        model = ElasticNet(
            alpha=trial.suggest_float(
                "alpha",
                1e-5,
                1e1,
                log=True,
            ),
            l1_ratio=trial.suggest_float(
                "l1_ratio",
                0.05,
                0.95,
            ),
            max_iter=20000,
            random_state=random_state,
        )


    # =====================================
    # Nonlinear models
    # =====================================

    elif model_name == "KernelRidge":

        model = KernelRidge(
            kernel="rbf",
            alpha=trial.suggest_float(
                "alpha",
                1e-4,
                1e4,
                log=True,
            ),
            gamma=trial.suggest_float(
                "gamma",
                1e-5,
                1e1,
                log=True,
            ),
        )


    elif model_name == "RandomForest":

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=random_state,
            n_jobs=1,

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                30,
            ),

            min_samples_leaf=trial.suggest_int(
                "min_samples_leaf",
                1,
                20,
            ),

            max_features=trial.suggest_float(
                "max_features",
                0.2,
                1.0,
                step=0.1,
            ),
        )


    elif model_name == "XGBoost":

        model = XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            learning_rate=0.05,
            random_state=random_state,
            n_jobs=1,

            max_depth=trial.suggest_int(
                "max_depth",
                2,
                10,
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.5,
                1.0,
                step=0.1,
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.5,
                1.0,
                step=0.1,
            ),

            min_child_weight=trial.suggest_float(
                "min_child_weight",
                1e-2,
                10,
                log=True,
            ),
        )


    elif model_name == "LightGBM":

        model = LGBMRegressor(
            objective="regression",
            n_estimators=300,
            learning_rate=0.05,
            random_state=random_state,
            n_jobs=1,
            verbosity=-1,

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                20,
            ),

            min_child_samples=trial.suggest_int(
                "min_child_samples",
                2,
                30,
            ),

            feature_fraction=trial.suggest_float(
                "feature_fraction",
                0.2,
                1.0,
                step=0.1,
            ),

            num_leaves=trial.suggest_int(
                "num_leaves",
                31,
                127,
            ),
        )


    else:
        raise ValueError(
            f"Unknown model name: {model_name}"
        )

    return model


# ============================================
# Generic Optuna tuner
# ============================================

def tune_model(
    model_name,
    prepared,
    n_trials=20,
    k_cv=10,
    random_state=0,
    n_jobs=10,
):

    x_train = prepared["x_train"]
    y_train = prepared["y_train"]

    cv, cv_groups = get_cv(
        prepared,
        k_cv=k_cv,
        random_state=random_state,
    )

    def objective(trial):

        model = get_tuned_model(
            model_name=model_name,
            trial=trial,
            prepared=prepared,
            random_state=random_state,
            n_jobs=n_jobs,
        )

        try:

            scores = model_selection.cross_validate(
                model,
                x_train,
                y_train,
                cv=cv,
                groups=cv_groups,
                scoring="r2",
                n_jobs=n_jobs,
                return_train_score=False,
                error_score=np.nan,
            )

            mean_score = np.nanmean(scores["test_score"])

            if np.isnan(mean_score):
                return -1e9

            return mean_score

        except Exception as e:

            logger.warning(
                "Trial failed: model=%s, error=%s",
                model_name,
                e,
            )

            return -1e9

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=random_state,
        ),
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True,
    )

    return {
        "study": study,
        "best_params": study.best_params,
        "best_cv_r2": study.best_value,
        "trial_df": collect_optuna_trials(study),
    }

In [20]:
# ============================================
# Run one tuned model
# ============================================

def run_tuned_model(
    species,
    model_name,
    prepared,
    n_trials=20,
    k_cv=10,
    random_state=0,
    n_jobs=10,
):
    tuning = tune_model(
        model_name=model_name,
        prepared=prepared,
        n_trials=n_trials,
        k_cv=k_cv,
        random_state=random_state,
        n_jobs=n_jobs,
    )

    best_trial = tuning["study"].best_trial

    model = get_tuned_model(
        model_name=model_name,
        trial=best_trial,
        prepared=prepared,
        random_state=random_state,
        n_jobs=n_jobs,
    )

    x_train = prepared["x_train"]
    x_test = prepared["x_test"]
    y_train = prepared["y_train"]
    y_test = prepared["y_test"]

    cv, cv_groups = get_cv(
        prepared,
        k_cv=k_cv,
        random_state=random_state,
    )

    best_cv_scores = model_selection.cross_validate(
        model,
        x_train,
        y_train,
        cv=cv,
        groups=cv_groups,
        scoring="r2",
        n_jobs=n_jobs,
        return_train_score=False,
        error_score=np.nan,
    )

    best_cv_r2_mean = np.nanmean(best_cv_scores["test_score"])
    best_cv_r2_sd = np.nanstd(best_cv_scores["test_score"])

    model.fit(x_train, y_train)

    pred_train = model.predict(x_train)
    pred_test = model.predict(x_test)

    train_metrics = evaluate_predictions(y_train, pred_train)
    test_metrics = evaluate_predictions(y_test, pred_test)

    return {
        "species": species,
        "model": model_name,
        "model_family": model_family_dict[model_name],
        "setting": "tuned",
        "CV_R2": best_cv_r2_mean,
        "CV_R2_sd": best_cv_r2_sd,
        "Train_R2": train_metrics["R2"],
        "Test_R2": test_metrics["R2"],
        "Train_Pearson_r": train_metrics["Pearson_r"],
        "Test_Pearson_r": test_metrics["Pearson_r"],
        "Train_RMSE": train_metrics["RMSE"],
        "Test_RMSE": test_metrics["RMSE"],
        "Train_MAE": train_metrics["MAE"],
        "Test_MAE": test_metrics["MAE"],
        "best_params": tuning["best_params"],
        "trial_df": tuning["trial_df"],
    }

In [21]:
# ============================================
# Run tuned benchmark: all species × all models
# ============================================

tuned_results = []

for species in species_config:
    for model_name in model_order:

        logger.info(
            "Running tuned model: species=%s, model=%s",
            species,
            model_name,
        )

        result = run_tuned_model(
            species=species,
            model_name=model_name,
            prepared=prepared_data[species],
            n_trials=n_trials,
            k_cv=k_cv,
            random_state=random_state,
            n_jobs=n_jobs,
        )

        tuned_results.append(result)

tuned_results_df = pd.DataFrame([
    {k: v for k, v in row.items() if k != "trial_df"}
    for row in tuned_results
])

display(tuned_results_df)

2026-06-18 16:51:28,210     INFO: Running tuned model: species=AT21, model=PLS
Best trial: 14. Best value: 0.16701: 100%|██████████| 20/20 [00:13<00:00,  1.51it/s]
2026-06-18 16:51:42,299     INFO: Running tuned model: species=AT21, model=Ridge
Best trial: 17. Best value: 0.178043: 100%|██████████| 20/20 [00:07<00:00,  2.84it/s]
2026-06-18 16:51:49,725     INFO: Running tuned model: species=AT21, model=Lasso
Best trial: 9. Best value: 0.181505: 100%|██████████| 20/20 [03:27<00:00, 10.36s/it]
2026-06-18 16:55:18,267     INFO: Running tuned model: species=AT21, model=ElasticNet
Best trial: 12. Best value: 0.182333: 100%|██████████| 20/20 [04:28<00:00, 13.42s/it]
2026-06-18 16:59:48,663     INFO: Running tuned model: species=AT21, model=KernelRidge
Best trial: 11. Best value: 0.219974: 100%|██████████| 20/20 [01:01<00:00,  3.05s/it]
2026-06-18 17:00:53,396     INFO: Running tuned model: species=AT21, model=RandomForest
Best trial: 9. Best value: 0.254756: 100%|██████████| 20/20 [17:06<00:

,species,model,model_family,setting,CV_R2,CV_R2_sd,Train_R2,Test_R2,Train_Pearson_r,Test_Pearson_r,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,best_params
0,AT21,PLS,Linear,tuned,0.167010,0.033207,0.275754,0.190609,0.525123,0.443101,0.851026,0.905102,0.657794,0.704225,{'n_components': 26}
1,AT21,Ridge,Linear,tuned,0.178043,0.031320,0.260004,0.198809,0.511346,0.447143,0.860230,0.900505,0.665015,0.704720,{'alpha': 319.923178995248}
2,AT21,Lasso,Linear,tuned,0.181505,0.030675,0.264091,0.200160,0.515073,0.448865,0.857851,0.899746,0.663129,0.704072,{'alpha': 0.001998246739232944}
3,AT21,ElasticNet,Linear,tuned,0.182333,0.030717,0.267860,0.199447,0.518455,0.448344,0.855652,0.900146,0.661121,0.703901,"{'alpha': 0.001677518022867653, 'l1_ratio': 0...."
4,AT21,KernelRidge,Nonlinear,tuned,0.219974,0.041638,0.463943,0.264865,0.689375,0.517704,0.732159,0.862584,0.563441,0.662385,"{'alpha': 0.0001466056399865723, 'gamma': 1.31..."
5,AT21,RandomForest,Nonlinear,tuned,0.254756,0.033670,0.576829,0.297555,0.782844,0.547742,0.650516,0.843188,0.495328,0.641270,"{'max_depth': 29, 'min_samples_leaf': 11, 'max..."
6,AT21,XGBoost,Nonlinear,tuned,0.241055,0.034606,0.537296,0.280060,0.751360,0.530780,0.680224,0.853623,0.525216,0.652880,"{'max_depth': 4, 'subsample': 0.6, 'colsample_..."
7,AT21,LightGBM,Nonlinear,tuned,0.246470,0.039295,0.617139,0.294829,0.807004,0.545113,0.618758,0.844822,0.476181,0.644520,"{'max_depth': 5, 'min_child_samples': 16, 'fea..."
8,NB21,PLS,Linear,tuned,0.225577,0.026836,0.361065,0.227468,0.600886,0.489237,0.799334,0.895329,0.618751,0.689102,{'n_components': 30}
9,NB21,Ridge,Linear,tuned,0.241434,0.026897,0.328107,0.212692,0.574701,0.465980,0.819691,0.903850,0.633068,0.694416,{'alpha': 536.8058219882244}


In [23]:
# ============================================
# Combine benchmark results
# ============================================

benchmark_results_df = pd.concat(
    [
        default_results_df,
        tuned_results_df,
    ],
    ignore_index=True,
)

benchmark_results_df["species"] = pd.Categorical(
    benchmark_results_df["species"],
    categories=list(species_config.keys()),
    ordered=True,
)

benchmark_results_df["model"] = pd.Categorical(
    benchmark_results_df["model"],
    categories=model_order,
    ordered=True,
)

benchmark_results_df["setting"] = pd.Categorical(
    benchmark_results_df["setting"],
    categories=["default", "tuned"],
    ordered=True,
)

benchmark_results_df = benchmark_results_df.sort_values(
    [
        "species",
        "model_family",
        "model",
        "setting",
    ]
).reset_index(drop=True)


# ============================================
# Save benchmark results
# ============================================

RESULT_DIR = Path("../results/table")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

benchmark_file = RESULT_DIR / "benchmark.tsv"

benchmark_results_df.to_csv(
    benchmark_file,
    sep="\t",
    index=False,
)

print(benchmark_file)
print(benchmark_file.exists())

# benchmark_excel_file = RESULT_DIR / "benchmark.xlsx"

# benchmark_results_df.to_excel(
#     benchmark_excel_file,
#     index=False,
# )

# print(benchmark_excel_file)
# print(benchmark_excel_file.exists())

../results/table/benchmark.tsv
True
